In [ ]:
import os, json, time, shutil, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
INPUT, OUTPUT = TASK / 'input' / PROFILE, TASK / 'output' / PROFILE
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
cfg = json.loads((INPUT / 'config.json').read_text())
SEED = cfg['seed']
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(cfg['threads'])

In [ ]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [ ]:
queries = pd.read_csv(INPUT / 'queries.csv').query("split == 'test'")
values = np.load(INPUT / 'query_fingerprints.npz')['values']
with np.load(INPUT / 'source_fingerprints.npz') as data:
    panels = {key: data[key] for key in data.files}
scale = np.load(INPUT / 'block_scale.npy')
fits = pd.read_csv(INPUT / 'fitted_parameters.csv')

In [ ]:
# no test ancestry is read in this task
rows, labels, probabilities, mixtures = [], [], [], []
for fit in fits.to_dict('records'):
    panel, markers, steps = fit['panel'], int(fit['markers']), int(fit['steps'])
    references = panels['exact_test'] if panel == 'exact' else panels['proxy']
    repeats = references.shape[1]
    centers = references.mean(axis=1)[:, :, :markers]
    direction = centers[0] - centers[1]
    denominator = np.maximum((direction ** 2).sum(axis=1), 1e-30)
    for query in queries.query('steps == @steps').to_dict('records'):
        observed = values[query['query_id']]
        cost = copying_cost(observed, references, markers, scale)
        local = cost.argmin(axis=1) // repeats
        global_path = np.full(len(cost), cost.sum(axis=0).argmin() // repeats)
        copied = viterbi(cost, fit['lambda']) // repeats
        local_p = source_probabilities(softmax(-cost / fit['temperature'], axis=1))
        copying_p = source_probabilities(posterior(cost, fit['lambda'], fit['temperature']))
        global_p = np.tile(source_probabilities(softmax(-cost.sum(axis=0) / fit['temperature'])), (len(cost), 1))
        alpha = np.clip(((observed[:, :markers] - centers[1]) * direction).sum(axis=1) / denominator, 0, 1)
        for method, path, p in [('global', global_path, global_p), ('nearest', local, local_p), ('copying', copied, copying_p)]:
            rows.append({'prediction_id': len(rows), 'panel': panel, 'markers': markers, 'method': method, **query})
            labels.append(path)
            probabilities.append(p)
            mixtures.append(alpha)
np.savez_compressed(OUTPUT / 'predictions.npz', labels=np.stack(labels), probabilities=np.stack(probabilities),
    mixture_weight_source_a=np.stack(mixtures))
pd.DataFrame(rows).to_csv(OUTPUT / 'prediction_index.csv', index=False)
display(pd.DataFrame(rows).groupby(['panel', 'markers', 'method']).size().rename('predictions').to_frame())